### Physics Informed DeepONet for an integrator problem

As an introduction to *Operator Learning*, we consider the problem of learning an integrator operator using a Physics Informed DeepONet. This is a simple 1-dimensional problem where the goal is to learn the mapping $f \mapsto u$, where $u(x) = \int_0^x f(s) \, ds$.

In this setup, the input to the DeepONet is a function $f$ and a point $x$, and the output is the value of the integral $u(x)$. The network is trained to satisfy the integral relation by minimizing a physics-informed loss function.

In [1]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

path = "/localdata/tomfre/SolutionData/Integrator/"

In [2]:
import qewton

F = qewton.config.Variable("f", 1)
X = qewton.config.Variable("x", 1)
U = qewton.config.Variable("u", 1)

In [3]:
f_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "f_data.pt")
u_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "u_data.pt")
x_data = qewton.DEFAULT_DL_BACKEND.load_data(path + "x_data.pt")

train_f_data, train_u_data, train_x_data = f_data[:4000], u_data[:4000], x_data[:4000]
test_f_data, test_u_data, test_x_data = f_data[4000:], u_data[4000:], x_data[4000:]

batch_size = 1000

In [4]:
dataset = qewton.data.GridDataSet(
    data=[train_f_data, train_x_data, train_u_data], 
    feature_variables=[F, X, U],
    geometry_variable=X,
    point_grid=x_data[0]
)

test_dataset = qewton.data.GridDataSet(
    data=[test_f_data, test_x_data, test_u_data],
    feature_variables=[F, X, U],
    geometry_variable=X,
    point_grid=x_data[0]
)

data_loader = qewton.data.DataLoader(
    data_set=dataset,
    test_data_set=test_dataset,
    batch_size=batch_size,
    splitting_ratio=(1.0, 0.0, 0.0),
    shuffle_data=False,
)

In [5]:
model = qewton.algorithms.FCNDeepONet(
    trunk_input=X, 
    branch_input=F,
    output=U,
    trunk_hidden_neurons=16,
    branch_hidden_neurons=16,
    trunk_hidden_layers=3,
    branch_hidden_layers=2,
    intermediate_neurons=32,
    activations=qewton.bb.Tanh
)

In [6]:
def ode_residual(x: X, f: F, u: U):
    return u.gradient(x) - f

def initial_residual(u: U):
    return u[:, 0]

ode_constraint = qewton.constraints.MSEConstraint()
initial_constraint = qewton.constraints.MSEConstraint(weight=50.0, name="initial_constraint")

pipeline = qewton.Graph()


with pipeline.tracker(0):
    f, x, u = data_loader()
    x = x.track_gradient()
    model_out = model(f, x)
    ode_res = ode_residual(x, f, model_out)
    initial_res = initial_residual(model_out)
    ode_constraint(ode_res, 0.0)
    initial_constraint(initial_res, 0.0)

In [7]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.01,
    max_iterations=2000,
    lr_scheduler=qewton.optim.StepLR(step_size=2000, gamma=0.25)
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[pipeline],
    training_objectives=[ode_constraint, initial_constraint],
    device=qewton.cuda(0),
)

trainer.run()

Optimization Phase 1: 100%|██████████| 2000/2000 [00:55<00:00, 36.10it/s, loss=0.0136] 


In [8]:
layout = pipeline.visualize(
    model.output_ports[0],
    reference=data_loader.get_output_port(U),
    mode=qewton.optim.EvaluationPhase.TEST,
)
fig = qewton.viz.Figure(layout)
fig.show()